# Validation empirique de la greffe gatée — version GPU (E1, E2, E3)

Version GPU de `notebook/experiments_surgery.ipynb` (CPU) : mêmes trois
expériences (E1 bit-exactness, E2 plasticité/point de selle, E3 localité du
coût vs profondeur), sur `NeuroDSL.Backend.CUDADevice()`.

**Deux vrais bugs de dispatch trouvés et corrigés en portant ce notebook sur
GPU** (pas des artefacts du notebook, des limitations réelles de
`src/dispatch.jl`/`src/backward.jl` jamais exercées sur GPU auparavant) :
- `dispatch.jl` (`:cross_entropy` forward) : `output_buffer[1] = loss` est un
  `setindex!` scalaire sur un `CuArray`, interdit par GPUArrays.jl sans
  `allowscalar`. Remplacé par `output_buffer .= loss` (broadcast, GPU-safe,
  même résultat).
- `backward.jl` (`:cross_entropy` backward) : `dlogits .*= dy[1]` est un
  `getindex` scalaire. Remplacé par `dlogits .*= dy` (broadcast sur un
  tableau de forme `(1,)`, identique numériquement).

**Autre écart GPU-spécifique** : `Backend.rand32(::CUDADevice,...)` délègue à
`CUDA.rand`, qui utilise l'état RNG **de CURAND**, pas le RNG global Julia
(`Random.seed!` n'a donc aucun effet sur l'initialisation des poids ici) --
`CUDA.seed!(...)` est appelé en plus, à chaque construction de modèle/greffe,
pour rester reproductible.

**Discipline de mesure GPU (E3)** : les kernels CUDA sont asynchrones --
chronométrer un lancement sans `CUDA.synchronize()` mesurerait le temps de
soumission, pas d'exécution. Chaque région chronométrée est encadrée d'une
synchronisation, en plus d'un warm-up dédié (compilation JIT CUDA, à part de
tout chiffre publié).

In [1]:
using NeuroDSL, Random, Printf, Statistics, DelimitedFiles, CUDA

const MASTER_SEED = 20260706
const DEV = NeuroDSL.Backend.CUDADevice()

println("GPU : ", CUDA.name(CUDA.device()))
println("VRAM libre : ", round(CUDA.free_memory() / 1024^3, digits=2), " / ",
        round(CUDA.total_memory() / 1024^3, digits=2), " Go")

const CFG = (
    d_model  = 128,
    n_heads  = 4,
    n_blocks = 4,
    vocab    = 50,
    seq_len  = 32,
    k_insert = 2,
    steps    = 600,
    lr       = 1f-3,
    tau_escape = 5,
)

results_dir = joinpath(@__DIR__, "results_surgery_gpu")
mkpath(results_dir)
println("Master seed : ", MASTER_SEED, "  (fixé avant toute run -- aucun filtrage post-hoc)")

GPU : NVIDIA RTX A5500 Laptop GPU
VRAM libre : 14.88 / 16.0 Go
Master seed : 20260706  (fixé avant toute run -- aucun filtrage post-hoc)


## Section 0 — Adaptateur NeuroDSL (GPU)

Identique en structure à la version CPU, seul le device change. `Random.seed!`
reste appelé (contrôle la génération des séquences via `rand(rng,...)`, qui
lui reste sur le RNG global Julia) ; `CUDA.seed!` est ajouté spécifiquement
pour l'initialisation des poids (`Backend.rand32`).

In [2]:
mutable struct ExpModel
    g::NeuroDSL.NeuroGraph
    ns::Symbol
    logits_sym::Symbol
    dim::Int
    n_heads::Int
    hidden_dim::Int
    m1::Dict{Symbol,AbstractArray{Float32}}   # CuArray sur GPU -- Array{Float32} n'accepterait pas ça
    m2::Dict{Symbol,AbstractArray{Float32}}
    t::Ref{Int}
    last_grads::Dict{Symbol,Array{Float32}}   # toujours CPU : copie explicite via Array(...) pour lecture diagnostique
end

function build_model(cfg::NamedTuple, rng::AbstractRNG)
    ns  = gensym(:expmodel)
    seed = rand(rng, 1:10^9)
    Random.seed!(seed); CUDA.seed!(seed)   # poids GPU via CURAND -- pas le RNG global
    g = NeuroDSL.NeuroGraph(namespace=ns, device=DEV)
    NeuroDSL.set!(g, :token_ids, ones(Int, cfg.seq_len); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.set!(g, :pos_ids, collect(1:cfg.seq_len); atom_type=NeuroDSL.Datom, namespace=ns)
    tok_emb = NeuroDSL.Embedding(cfg.vocab, cfg.d_model)(g, :token_ids, :tok; namespace=ns)
    pos_emb = NeuroDSL.Embedding(cfg.seq_len, cfg.d_model)(g, :pos_ids, :pos; namespace=ns)
    xsum = :embed_sum
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(xsum, [tok_emb, pos_emb], :add; namespace=ns))
    hidden_dim = 4 * cfg.d_model
    out = NeuroDSL.LlamaModel(cfg.n_blocks, cfg.d_model, cfg.n_heads, hidden_dim)(g, xsum; namespace=ns)
    logits = NeuroDSL.Linear(cfg.d_model, cfg.vocab)(g, out, :lm_head; namespace=ns)
    NeuroDSL.set!(g, :labels, ones(Int, cfg.seq_len); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [logits, :labels], :cross_entropy; namespace=ns))
    return ExpModel(g, ns, logits, cfg.d_model, cfg.n_heads, hidden_dim,
                     Dict{Symbol,AbstractArray{Float32}}(), Dict{Symbol,AbstractArray{Float32}}(), Ref(0),
                     Dict{Symbol,Array{Float32}}())
end

function forward_logits(model::ExpModel, X)
    NeuroDSL.set!(model.g, :token_ids, X; atom_type=NeuroDSL.Datom, namespace=model.ns)
    NeuroDSL.invalidate_all!(model.g; namespace=model.ns)
    return Array(NeuroDSL.demand!(model.g, model.logits_sym; namespace=model.ns))
end

function graft_shadow_block!(model::ExpModel, k::Int; alpha0::Float32,
                             zero_out_proj::Bool, rng::AbstractRNG)
    seed = rand(rng, 1:10^9)
    Random.seed!(seed); CUDA.seed!(seed)
    after_sym = Symbol(:layer_, k, :_out)
    _, handle = NeuroDSL.graft_shadow_block!(model.g, model.ns, after_sym,
                                              model.dim, model.n_heads, model.hidden_dim;
                                              alpha0=alpha0, zero_out_proj=zero_out_proj)
    NeuroDSL.invalidate_all!(model.g; namespace=model.ns)
    for p in NeuroDSL.params(model.g; namespace=model.ns)
        haskey(model.m1, p.name) && continue
        model.m1[p.name] = NeuroDSL.Backend.zeros32(DEV, size(p.value)...)
        model.m2[p.name] = NeuroDSL.Backend.zeros32(DEV, size(p.value)...)
    end
    return merge(handle, (; model))
end

function branch_output(handle, model::ExpModel, X)
    NeuroDSL.set!(model.g, :token_ids, X; atom_type=NeuroDSL.Datom, namespace=model.ns)
    NeuroDSL.invalidate_all!(model.g; namespace=model.ns)
    return Array(NeuroDSL.demand!(model.g, handle.R_sym; namespace=model.ns))
end

function residual_stream_at(model::ExpModel, k::Int, X)
    after_sym = Symbol(:layer_, k, :_out)
    NeuroDSL.set!(model.g, :token_ids, X; atom_type=NeuroDSL.Datom, namespace=model.ns)
    NeuroDSL.invalidate_all!(model.g; namespace=model.ns)
    return Array(NeuroDSL.demand!(model.g, after_sym; namespace=model.ns))
end

function train_step!(model::ExpModel, X, Y; lr::Float32)::Float32
    model.t[] += 1
    NeuroDSL.set!(model.g, :token_ids, X; atom_type=NeuroDSL.Datom, namespace=model.ns)
    NeuroDSL.set!(model.g, :labels, Y; atom_type=NeuroDSL.Datom, namespace=model.ns)
    NeuroDSL.invalidate_all!(model.g; namespace=model.ns)
    loss_val = NeuroDSL.demand!(model.g, :loss; namespace=model.ns)
    NeuroDSL.backward_graph!(model.g, :loss; namespace=model.ns)

    empty!(model.last_grads)
    for p in NeuroDSL.params(model.g; namespace=model.ns)
        p.gradient === nothing && continue
        model.last_grads[p.name] = copy(Array(p.gradient))
    end

    for p in NeuroDSL.params(model.g; namespace=model.ns)
        p.gradient === nothing && continue
        if !haskey(model.m1, p.name)
            model.m1[p.name] = NeuroDSL.Backend.zeros32(DEV, size(p.value)...)
            model.m2[p.name] = NeuroDSL.Backend.zeros32(DEV, size(p.value)...)
        end
        NeuroDSL.adamw_step!(DEV, p.value, p.gradient,
                              model.m1[p.name], model.m2[p.name],
                              lr, 0.9f0, 0.999f0, 1f-8, model.t[], 1f0, 0f0)
    end
    loss_scalar = Float32(sum(Array(loss_val)))
    NeuroDSL.invalidate_all!(model.g; namespace=model.ns)
    return loss_scalar
end

function branch_grad_norm(handle)::Float64
    model = handle.model
    mha = Symbol(handle.prefix, :_mha)
    theta_syms = [Symbol(mha,:_q_W), Symbol(mha,:_k_W), Symbol(mha,:_v_W), Symbol(mha,:_output_W),
                  Symbol(handle.prefix,:_mlp_w1), Symbol(handle.prefix,:_mlp_w2), Symbol(handle.prefix,:_mlp_w3)]
    total = 0.0
    for sym in theta_syms
        gr = get(model.last_grads, sym, nothing)
        gr === nothing && continue
        total += sum(abs2, gr)
    end
    return sqrt(total)
end

alpha_value(handle)::Float32 = Array(NeuroDSL.node(handle.model.g, handle.alpha_sym; namespace=handle.model.ns).value)[1]
function alpha_grad(handle)::Float32
    gr = get(handle.model.last_grads, handle.alpha_sym, nothing)
    return gr === nothing ? 0f0 : gr[1]
end

function eval_loss(model::ExpModel, X, Y)::Float32
    NeuroDSL.set!(model.g, :token_ids, X; atom_type=NeuroDSL.Datom, namespace=model.ns)
    NeuroDSL.set!(model.g, :labels, Y; atom_type=NeuroDSL.Datom, namespace=model.ns)
    NeuroDSL.invalidate_all!(model.g; namespace=model.ns)
    return Float32(sum(Array(NeuroDSL.demand!(model.g, :loss; namespace=model.ns))))
end

function make_batch(rng::AbstractRNG, cfg)
    return NeuroDSL.sample_induction_sequence(rng, cfg.vocab, cfg.seq_len ÷ 2)
end

println("Section 0 (adaptateur GPU) chargée.")

Section 0 (adaptateur GPU) chargée.


## Section 1 — E1 : bit-exactness (GPU)

In [3]:
function check_preconditions(model, handle, X, k)
    res = residual_stream_at(model, k, X)
    r   = branch_output(handle, model, X)
    n_negzero = count(v -> iszero(v) && signbit(v), res)
    n_nonfin  = count(!isfinite, r)
    ok  = (n_negzero == 0) && (n_nonfin == 0)
    rep = @sprintf("""
    Préconditions (Prop. 2) :
      composantes -0.0 dans le flux résiduel : %d / %d   %s
      valeurs non finies dans R(x)           : %d / %d   %s
    """, n_negzero, length(res), n_negzero == 0 ? "[OK]" : "[VIOLATION]",
         n_nonfin, length(r),   n_nonfin == 0 ? "[OK]" : "[VIOLATION]")
    return ok, rep
end

function run_E1(master_seed)
    println("="^70); println("E1 — BIT-EXACTNESS (GPU)"); println("="^70)
    rng_model = MersenneTwister(hash((master_seed, :model)))
    rng_data  = MersenneTwister(hash((master_seed, :data)))
    rng_graft = MersenneTwister(hash((master_seed, :graft)))

    model = build_model(CFG, rng_model)
    X, _  = make_batch(rng_data, CFG)

    y_before = forward_logits(model, X)
    bits_before = reinterpret(UInt32, vec(y_before))

    handle = graft_shadow_block!(model, CFG.k_insert; alpha0=0f0, zero_out_proj=false, rng=rng_graft)
    pre_ok, pre_report = check_preconditions(model, handle, X, CFG.k_insert)
    print(pre_report)

    y_after = forward_logits(model, X)
    bits_after = reinterpret(UInt32, vec(y_after))
    n_mismatch = count(bits_before .!= bits_after)
    n_total    = length(bits_before)
    verdict = n_mismatch == 0 ? "BIT-EXACT [OK]" :
              (y_before ≈ y_after ? "fonctionnellement égal mais PAS bit-exact [X]" : "NON EXACT [XX]")

    report = @sprintf("""
    %s
    Éléments comparés : %d
    Mismatches bit    : %d  (%.2e %%)
    Max |Δ| flottant  : %.3e
    Préconditions     : %s
    Verdict           : %s
    """, "-"^50, n_total, n_mismatch, 100n_mismatch/n_total,
         maximum(abs.(y_before .- y_after)),
         pre_ok ? "satisfaites" : "VIOLÉES", verdict)
    println(report)
    write(joinpath(results_dir, "E1_report.txt"), pre_report * report)
    return n_mismatch == 0
end

ok1 = run_E1(MASTER_SEED)

E1 — BIT-EXACTNESS (GPU)
✅ Op :scalar_gate registered
Préconditions (Prop. 2) :
  composantes -0.0 dans le flux résiduel : 0 / 4096   [OK]
  valeurs non finies dans R(x)           : 0 / 4096   [OK]
--------------------------------------------------
Éléments comparés : 1600
Mismatches bit    : 0  (0.00e+00 %)
Max |Δ| flottant  : 0.000e+00
Préconditions     : satisfaites
Verdict           : BIT-EXACT [OK]



true

## Section 2 — E2 : plasticité vs point de selle (GPU, 3 bras)

In [4]:
const ARMS = (
    (name="rezero",     alpha0=0f0, zero_out=false),
    (name="net2net",    alpha0=1f0, zero_out=true),
    (name="degenerate", alpha0=0f0, zero_out=true),
)

function run_arm(arm, master_seed)
    rng_model = MersenneTwister(hash((master_seed, :model)))
    rng_data  = MersenneTwister(hash((master_seed, :data)))
    rng_graft = MersenneTwister(hash((master_seed, :graft)))

    model  = build_model(CFG, rng_model)
    handle = graft_shadow_block!(model, CFG.k_insert; alpha0=arm.alpha0, zero_out_proj=arm.zero_out, rng=rng_graft)

    hist = zeros(Float64, CFG.steps, 5)
    for t in 1:CFG.steps
        X, Y = make_batch(rng_data, CFG)
        loss = train_step!(model, X, Y; lr=CFG.lr)
        hist[t, :] .= (t, loss, abs(alpha_value(handle)), abs(alpha_grad(handle)), branch_grad_norm(handle))
    end

    path = joinpath(results_dir, "E2_$(arm.name).csv")
    open(path, "w") do io
        println(io, "step,loss,abs_alpha,abs_grad_alpha,grad_theta_norm")
        writedlm(io, hist, ',')
    end
    return hist
end

function run_E2(master_seed)
    println("\n"); println("="^70); println("E2 — PLASTICITÉ / POINT SELLE (GPU)"); println("="^70)
    results = Dict{String, Matrix{Float64}}()
    for arm in ARMS
        println("  bras `$(arm.name)`  (alpha0=$(arm.alpha0), zero_out_proj=$(arm.zero_out)) ...")
        results[arm.name] = run_arm(arm, master_seed)
    end

    io = IOBuffer()
    println(io, "-"^70)
    @printf(io, "%-12s | %10s | %10s | %12s | %12s\n", "bras", "loss t=1", "loss fin", "|alpha| fin", "‖∇θ‖ max")
    println(io, "-"^70)
    for arm in ARMS
        h = results[arm.name]
        @printf(io, "%-12s | %10.4f | %10.4f | %12.3e | %12.3e\n", arm.name, h[1,2], h[end,2], h[end,3], maximum(h[:,5]))
    end
    println(io, "-"^70)

    h_rz, h_dg = results["rezero"], results["degenerate"]
    checks = [
        ("Prop.4 — dL/dalpha != 0 au premier backward (rezero)", h_rz[1,4] > 0),
        ("Prop.4 — |alpha| > 0 en <= $(CFG.tau_escape) pas (rezero)", any(h_rz[1:CFG.tau_escape, 3] .> 0)),
        ("Prop.4 — ‖∇θ‖ devient non nul après l'escape (rezero)", maximum(h_rz[:,5]) > 0),
        ("Prop.3 — dL/dalpha == 0 sur toute la run (degenerate)", all(h_dg[:,4] .== 0)),
        ("Prop.3 — ‖∇θ‖ == 0 sur toute la run (degenerate)", all(h_dg[:,5] .== 0)),
        ("Prop.3 — alpha ne bouge jamais (degenerate)", all(h_dg[:,3] .== 0)),
        ("Net2Net — le bras (b) apprend (loss décroît)", results["net2net"][end,2] < results["net2net"][1,2]),
    ]
    println(io, "\nAssertions :")
    all_ok = true
    for (label, ok) in checks
        all_ok &= ok
        println(io, (ok ? "  [PASS] " : "  [FAIL] ") * label)
    end
    summary = String(take!(io))
    println(summary)
    write(joinpath(results_dir, "E2_summary.txt"), summary)
    return all_ok
end

ok2 = run_E2(MASTER_SEED)



E2 — PLASTICITÉ / POINT SELLE (GPU)
  bras `rezero`  (alpha0=0.0, zero_out_proj=false) ...
  bras `net2net`  (alpha0=1.0, zero_out_proj=true) ...
  bras `degenerate`  (alpha0=0.0, zero_out_proj=true) ...
----------------------------------------------------------------------
bras         |   loss t=1 |   loss fin |  |alpha| fin |     ‖∇θ‖ max
----------------------------------------------------------------------
rezero       |     3.8908 |     1.9926 |    4.644e-03 |    2.870e-02
net2net      |     3.8908 |     1.9182 |    7.408e-01 |    1.620e+00
degenerate   |     3.8908 |     1.9599 |    0.000e+00 |    0.000e+00
----------------------------------------------------------------------

Assertions :
  [PASS] Prop.4 — dL/dalpha != 0 au premier backward (rezero)
  [PASS] Prop.4 — |alpha| > 0 en <= 5 pas (rezero)
  [PASS] Prop.4 — ‖∇θ‖ devient non nul après l'escape (rezero)
  [PASS] Prop.3 — dL/dalpha == 0 sur toute la run (degenerate)
  [PASS] Prop.3 — ‖∇θ‖ == 0 sur toute la run (degene

true

## Section 3 — E3 : coût de la chirurgie vs profondeur (GPU)

Régime différent du CPU : chaque `demand!`/`patch` est une série de lancements
de kernels CUDA asynchrones. Chronométrer correctement exige (a) un
`CUDA.synchronize()` avant de démarrer le chrono (pour ne pas hériter d'un
travail encore en vol d'avant) et (b) un `CUDA.synchronize()` juste avant de
lire `time_ns()` à la fin (sinon on mesure le temps de *soumission*, pas
d'exécution). Discipline déjà établie dans ce projet pour les benchmarks GPU
(`gpu_scale_full.jl`, article 2, §gpuscale) : warm-up dédié par chemin de code,
horloge non verrouillée ici (mesure exploratoire, pas un chiffre d'article).

In [5]:
const CFG3 = (
    d_model  = 128,
    n_heads  = 4,
    n_blocks = 8,
    vocab    = 256,
    seq_len  = 32,
    n_reps   = 15,
    trim     = 0.10,
    warmup_reps = 5,   # un peu plus que CPU -- compilation JIT CUDA en plus de Julia
)
const DEPTHS = collect(1:(CFG3.n_blocks - 1))

trimmed_median(v, trim) = begin
    s = sort(v); n = length(s); cut = floor(Int, trim * n)
    median(s[(cut + 1):(n - cut)])
end

make_input(rng, cfg) = rand(rng, 1:cfg.vocab, cfg.seq_len)

function downstream_cone_size(model::ExpModel, k::Int)::Int
    after_sym = Symbol(:layer_, k, :_out)
    return length(NeuroDSL._downstream_nodes(model.g, after_sym, model.ns))
end
graph_size(model::ExpModel)::Int = length(model.g.nodes[model.ns])

function set_input!(model::ExpModel, X)
    NeuroDSL.set!(model.g, :token_ids, X; atom_type=NeuroDSL.Datom, namespace=model.ns)
end
function forward_logits_readonly(model::ExpModel)
    y = NeuroDSL.demand!(model.g, model.logits_sym; namespace=model.ns)
    CUDA.synchronize()
    return y
end
function graft_shadow_block_readonly!(model::ExpModel, k::Int; alpha0::Float32,
                                       zero_out_proj::Bool, rng::AbstractRNG)
    seed = rand(rng, 1:10^9)
    Random.seed!(seed); CUDA.seed!(seed)
    after_sym = Symbol(:layer_, k, :_out)
    _, handle = NeuroDSL.graft_shadow_block!(model.g, model.ns, after_sym,
                                              model.dim, model.n_heads, model.hidden_dim;
                                              alpha0=alpha0, zero_out_proj=zero_out_proj)
    CUDA.synchronize()
    return merge(handle, (; model))
end

function measure_one(k::Int, rep::Int, master_seed)
    rng_model = MersenneTwister(hash((master_seed, :model, k, rep)))
    rng_data  = MersenneTwister(hash((master_seed, :data,  k, rep)))
    rng_graft = MersenneTwister(hash((master_seed, :graft, k, rep)))

    model = build_model(CFG3, rng_model)
    X = make_input(rng_data, CFG3)
    set_input!(model, X)
    forward_logits_readonly(model)

    CUDA.synchronize()
    t0 = time_ns()
    graft_shadow_block_readonly!(model, k; alpha0=0f0, zero_out_proj=false, rng=rng_graft)
    CUDA.synchronize()
    t_graft_ms = (time_ns() - t0) / 1e6

    CUDA.synchronize()
    t1 = time_ns()
    forward_logits_readonly(model)
    t_recompute_ms = (time_ns() - t1) / 1e6

    return t_graft_ms, t_recompute_ms
end

function run_E3(master_seed)
    println("="^70); println("E3 — COÛT DE LA CHIRURGIE vs PROFONDEUR (GPU)"); println("="^70)
    println("Master seed : $master_seed   |   profondeurs : $(DEPTHS)")

    rng0  = MersenneTwister(hash((master_seed, :model, 0, 0)))
    rng0d = MersenneTwister(hash((master_seed, :data,  0, 0)))
    m0    = build_model(CFG3, rng0)
    set_input!(m0, make_input(rng0d, CFG3))
    forward_logits_readonly(m0)
    total = graph_size(m0)
    cones = Dict(k => downstream_cone_size(m0, k) for k in DEPTHS)
    println("\nGraphe : $total nœuds. Cônes aval par profondeur :")
    for k in DEPTHS
        @printf("  k=%d : %d nœuds (%.1f%%)\n", k, cones[k], 100cones[k]/total)
    end

    println("\nWarm-up ($(CFG3.warmup_reps) greffes jetables, compilation JIT CUDA)...")
    for w in 1:CFG3.warmup_reps
        measure_one(first(DEPTHS), -w, master_seed)
        measure_one(last(DEPTHS),  -w, master_seed)
    end

    graft_t  = Dict(k => Float64[] for k in DEPTHS)
    recomp_t = Dict(k => Float64[] for k in DEPTHS)
    for rep in 1:CFG3.n_reps
        for k in DEPTHS
            tg, tr = measure_one(k, rep, master_seed)
            push!(graft_t[k], tg); push!(recomp_t[k], tr)
        end
        rep % 5 == 0 && println("  ... répétition $rep/$(CFG3.n_reps)")
    end

    rows = zeros(Float64, length(DEPTHS), 5)
    for (i, k) in enumerate(DEPTHS)
        rows[i, :] .= (k, cones[k],
                       trimmed_median(graft_t[k],  CFG3.trim),
                       trimmed_median(recomp_t[k], CFG3.trim),
                       trimmed_median(graft_t[k],  CFG3.trim) + trimmed_median(recomp_t[k], CFG3.trim))
    end
    open(joinpath(results_dir, "E3_cost_vs_depth_gpu.csv"), "w") do io
        println(io, "depth_k,cone_size,graft_ms,recompute_ms,total_ms")
        writedlm(io, rows, ',')
    end

    io = IOBuffer()
    println(io, "-"^70)
    @printf(io, "%4s | %9s | %10s | %13s | %10s\n", "k", "cône", "greffe(ms)", "recompute(ms)", "total(ms)")
    println(io, "-"^70)
    for i in 1:size(rows, 1)
        @printf(io, "%4d | %9d | %10.3f | %13.3f | %10.3f\n",
                Int(rows[i,1]), Int(rows[i,2]), rows[i,3], rows[i,4], rows[i,5])
    end
    println(io, "-"^70)

    recomps = rows[:, 4]; csizes = rows[:, 2]
    r = cor(csizes, recomps)
    mono_ok = all(recomps[i] >= recomps[i+1] * 0.90 for i in 1:length(recomps)-1)
    checks = [
        ("Localité — coût de recompute décroît (monotone, tol. bruit 10%) avec k", mono_ok),
        ("Localité — corrélation recompute vs taille de cône r > 0.95 (r = $(round(r, digits=4)))", r > 0.95),
        ("Localité — coût k=$(last(DEPTHS)) < 50% du coût k=$(first(DEPTHS))", recomps[end] < 0.5 * recomps[1]),
    ]
    println(io, "\nAssertions :")
    all_ok = true
    for (label, ok) in checks
        all_ok &= ok
        println(io, (ok ? "  [PASS] " : "  [FAIL] ") * label)
    end
    println(io, """

    Lecture GPU vs CPU : sous Windows/WDDM, chaque lancement de kernel coûte un
    surcoût fixe (~0.01-0.02 ms, déjà mesuré dans ce projet pour la
    restauration par cache -- voir src/patching.jl, section "Restauration GPU
    par lots"). À cette échelle (quelques centaines de nœuds), ce surcoût peut
    dominer le calcul réel -- si le gain GPU par rapport au CPU est faible ou
    négatif ici, ce n'est pas une réfutation de la localité (la corrélation
    coût/cône ci-dessus est la vraie preuve), c'est une question d'échelle : le
    GPU n'amortit son propre coût fixe qu'à partir d'un cône bien plus large.
    """)
    summary = String(take!(io))
    println(summary)
    write(joinpath(results_dir, "E3_summary_gpu.txt"), summary)
    return all_ok
end

ok3 = run_E3(MASTER_SEED)

E3 — COÛT DE LA CHIRURGIE vs PROFONDEUR (GPU)
Master seed : 20260706   |   profondeurs : [1, 2, 3, 4, 5, 6, 7]

Graphe : 412 nœuds. Cônes aval par profondeur :
  k=1 : 289 nœuds (70.1%)
  k=2 : 248 nœuds (60.2%)
  k=3 : 207 nœuds (50.2%)
  k=4 : 166 nœuds (40.3%)
  k=5 : 125 nœuds (30.3%)
  k=6 : 84 nœuds (20.4%)
  k=7 : 43 nœuds (10.4%)

Warm-up (5 greffes jetables, compilation JIT CUDA)...
  ... répétition 5/15
  ... répétition 10/15
  ... répétition 15/15
----------------------------------------------------------------------
   k |      cône | greffe(ms) | recompute(ms) |  total(ms)
----------------------------------------------------------------------
   1 |       289 |      1.415 |        10.604 |     12.018
   2 |       248 |      1.403 |         9.654 |     11.057
   3 |       207 |      1.525 |         9.639 |     11.163
   4 |       166 |      1.418 |         7.742 |      9.160
   5 |       125 |      1.549 |         6.025 |      7.574
   6 |        84 |      1.607 |         4

true

In [6]:
println("="^70)
println(ok1 && ok2 && ok3 ? ">>> TOUTES LES CLAIMS VALIDÉES SUR GPU -- résultats dans results_surgery_gpu/."
                          : ">>> AU MOINS UN ÉCHEC -- voir results_surgery_gpu/ ; ne PAS publier avant résolution.")

>>> TOUTES LES CLAIMS VALIDÉES SUR GPU -- résultats dans results_surgery_gpu/.
